In [532]:
import pandas as pd
import numpy as np 

In [533]:
dataset1 = pd.read_csv('../data/used_cars_data.csv')
dataset2 = pd.read_csv('../data/used_car_dataset.csv')

Droping Unnessery column from the dataset 1

In [534]:
Drop_variables_dataset1 =['S.No.','Location','Engine','Power','Seats','New_Price','Mileage']
dataset1 = dataset1.drop(Drop_variables_dataset1, axis=1)

Spliting Name as model and brand


In [535]:
dataset1[['Brand','Model']] = dataset1['Name'].str.split(' ',n=1 ,expand = True)
dataset1 =dataset1.drop(['Name'],axis=1)
# rearranging the column 
cols = list(dataset1.columns)
cols.remove('Brand')
cols.remove('Model')
new_order = ['Brand', 'Model'] + cols
dataset1 = dataset1[new_order]

In [536]:
dataset1 = dataset1.iloc[:-1234, :]
#print(dataset1.isna().sum())
dataset1['Brand'] = dataset1['Brand'].replace('Land', 'Land Rover')
dataset1['Brand'] = dataset1['Brand'].replace('Maruti', 'maruti suzuki')

In [537]:
alphabetic_columns = ['Brand','Fuel_Type','Transmission','Owner_Type']
dataset1[alphabetic_columns] = dataset1[alphabetic_columns].map(lambda x: x.lower() if isinstance(x, str) else x)

In [538]:
dataset2 = dataset2.drop(['PostedDate','AdditionInfo','Age'],axis=1)

In [539]:
dataset2['AskPrice'] = dataset2['AskPrice'].str.replace('₹', '', regex=False)
dataset2['AskPrice'] = dataset2['AskPrice'].str.replace(',', '', regex=False)
dataset2['AskPrice'] = dataset2['AskPrice'].str.strip()
dataset2['AskPrice'] = pd.to_numeric(dataset2['AskPrice'], errors='coerce')
dataset1['Price'] = pd.to_numeric(dataset1['Price'],errors='coerce')
dataset2['AskPrice'] = dataset2['AskPrice'] / 100000  # convert to Lakhs
dataset2.rename(columns={'AskPrice': 'Price'}, inplace=True)

In [540]:
dataset2.rename(columns={
    'model': 'Model',
    'kmDriven': 'Kilometers_Driven',
    'FuelType': 'Fuel_Type',
    'Owner': 'Owner_Type'
}, inplace=True)
dataset2[alphabetic_columns] = dataset2[alphabetic_columns].map(lambda x: x.lower() if isinstance(x, str) else x)


In [541]:
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].str.replace(',', '', regex=False)
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].str.replace(' km', '', regex=False)
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].str.strip()
dataset2['Kilometers_Driven'] = pd.to_numeric(dataset2['Kilometers_Driven'], errors='coerce')

In [542]:
dataset2['Kilometers_Driven'] = dataset2['Kilometers_Driven'].replace(0,pd.NA)

In [543]:
cols = ['Brand', 'Model', 'Year', 'Kilometers_Driven', 'Transmission', 'Owner_Type', 'Fuel_Type', 'Price']
dataset1_clean =dataset1[cols]
dataset2_clean =dataset1[cols]

dataset_merged = pd.concat([dataset1_clean , dataset2_clean], ignore_index=True)

In [544]:
from sklearn.impute import SimpleImputer
si  = SimpleImputer(missing_values=np.nan , strategy='median')
dataset_merged['Kilometers_Driven'] = si.fit_transform(dataset_merged[['Kilometers_Driven']])

In [545]:
dataset_merged= dataset_merged[dataset_merged['Kilometers_Driven'] < 300000]
dataset_merged = dataset_merged[dataset_merged['Price'] < 100]
print('Before drop_duplicates:', 12024)
dataset_merged = dataset_merged.drop_duplicates()
print('After drop_duplicates:', dataset_merged.shape[0])
print('Removed:', 12024 - dataset_merged.shape[0])
print(dataset_merged.shape)

Before drop_duplicates: 12024
After drop_duplicates: 6008
Removed: 6016
(6008, 8)


In [546]:
brand_counts = dataset_merged['Brand'].value_counts()
rare_brands = brand_counts[brand_counts < 10].index
dataset_merged['Brand'] = dataset_merged['Brand'].replace(rare_brands, 'other')

In [547]:
fuel_counts = dataset_merged['Fuel_Type'].value_counts()
rare_fuels = fuel_counts[fuel_counts < 21].index
dataset_merged['Fuel_Type'] = dataset_merged['Fuel_Type'].replace(rare_fuels, 'other')
print(dataset_merged['Owner_Type'].unique())


<StringArray>
['first', 'second', 'fourth & above', 'third']
Length: 4, dtype: str


In [548]:
from sklearn.preprocessing import OrdinalEncoder
o_encoder = OrdinalEncoder(categories=[['fourth & above', 'third', 'second', 'first']])
dataset_merged['Owner_Type'] =o_encoder.fit_transform(dataset_merged[['Owner_Type']])
print(dataset_merged['Owner_Type'].unique())

[3. 2. 0. 1.]


In [549]:

dataset_merged= dataset_merged.drop(['Model'],axis=1)

In [550]:
from sklearn.preprocessing import LabelEncoder 
le = LabelEncoder()
dataset_merged['Transmission'] = le.fit_transform(dataset_merged['Transmission']) 

In [551]:
dataset_merged= pd.get_dummies(dataset_merged, columns=['Brand', 'Fuel_Type'], drop_first=True).astype(int)


In [552]:
print(dataset_merged.dtypes)

Year                   int64
Kilometers_Driven      int64
Transmission           int64
Owner_Type             int64
Price                  int64
Brand_bmw              int64
Brand_chevrolet        int64
Brand_datsun           int64
Brand_fiat             int64
Brand_ford             int64
Brand_honda            int64
Brand_hyundai          int64
Brand_jaguar           int64
Brand_jeep             int64
Brand_land rover       int64
Brand_mahindra         int64
Brand_maruti suzuki    int64
Brand_mercedes-benz    int64
Brand_mini             int64
Brand_mitsubishi       int64
Brand_nissan           int64
Brand_other            int64
Brand_porsche          int64
Brand_renault          int64
Brand_skoda            int64
Brand_tata             int64
Brand_toyota           int64
Brand_volkswagen       int64
Brand_volvo            int64
Fuel_Type_diesel       int64
Fuel_Type_other        int64
Fuel_Type_petrol       int64
dtype: object


In [553]:
dataset_merged.to_csv('../data/cleaned_data.csv', index=False)